# 02. A/B/C 검색 테스트 (같은 질문 · 같은 채점)

| 안 | 구성 | 함수 |
|---|---|---|
| A | Qdrant 벡터 검색 | `retrieve_a` |
| B | Neo4j 벡터 검색 | `retrieve_b` |
| C | Qdrant·Neo4j 벡터 검색 결과 결합 | `retrieve_c` |

세 안은 A_fixed v1 벡터와 같은 KURE-v1 질문 임베딩을 쓴다.
입력은 v3 122개, abc 20개, ragas 32개다. 채점은 `abc_eval.py`에서 정책·섹션 부모 ID 기준으로 진행한다.
시설 검색은 벡터 평가에서 제외하며, 답변 기권 여부가 필요한 함정은 검색 결과만으로 채점하지 않는다.

## 1. 준비

- 프로젝트 가상환경을 노트북 커널로 선택한다. 누락된 패키지는 저장소 루트 터미널에서 `uv pip install --python .venv/Scripts/python.exe qdrant-client langchain-qdrant langchain-huggingface sentence-transformers`로 설치한다.
- 저장소 루트의 Compose로 Qdrant와 Neo4j를 실행한다. Qdrant는 [01 적재 노트북](01_qdrant_setup.ipynb) 또는 [스냅샷 복원 안내](../Qdrant_test/welfare_chunks_v1_snapshot_20261110/RESTORE_GUIDE.md)로 준비한다.
- Neo4j는 [적재 및 덤프 복원 안내](../Neo4j_test/neo4j_description.md)에 따라 준비한다.
- 루트 `.env`에서 Qdrant는 `QDRANT_URL`, `QDRANT_COLLECTION`, 선택 항목 `QDRANT_API_KEY`를 읽는다. Neo4j는 `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, 선택 항목 `NEO4J_DATABASE`를 읽는다.
- A만 평가하려면 아래 두 옵션을 모두 `False`로 설정한다. 처음 실행할 때는 KURE-v1 모델을 내려받는다.

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

USE_B = True
USE_C = True

cwd = Path.cwd().resolve()
kit_root = next(
    (
        candidate
        for directory in (cwd, *cwd.parents)
        for candidate in (directory, directory / "notebooks")
        if (candidate / "abc_eval.py").is_file()
        and (candidate / "data" / "questions").is_dir()
    ),
    None,
)
if kit_root is None:
    raise FileNotFoundError("프로젝트 폴더 안에서 노트북을 실행하세요. 평가 키트를 찾지 못했습니다.")

project_root = kit_root.parent
for import_path in (str(project_root), str(kit_root)):
    if import_path not in sys.path:
        sys.path.insert(0, import_path)
load_dotenv(project_root / ".env")

import abc_eval as E

data_dir = kit_root / "data"
embedding_info = json.loads((data_dir / "neo4j" / "embedding_info.json").read_text(encoding="utf-8"))
question_dir = data_dir / "questions"
questions = (
    E.load_questions(question_dir / "test_questions_v3.jsonl", "v3")
    + E.load_questions(question_dir / "test_questions_abc_v1.jsonl", "abc")
    + E.load_questions(question_dir / "test_questions_ragas_v1.jsonl", "ragas")
)
print("전체 질문:", len(questions))
print(pd.Series([(q["set"], q["mode"]) for q in questions]).value_counts())

# 광역기관 소관 여부는 적재된 정책의 담당 부서를 사용한다.
# 시군구 값이 빠진 정책도 담당 부서가 시·군·구면 광역 후보에서 제외한다.
province_policies = {}
with (data_dir / "chunks/A_fixed/policy_chunks.jsonl").open(encoding="utf-8") as stream:
    for line in stream:
        row = json.loads(line)
        meta = row["metadata"]
        sido = meta.get("sido_name")
        if not sido or meta.get("sigungu_name"):
            continue
        department = (meta.get("department") or "").split()
        province_department = (
            len(department) > 1
            and department[0] == sido
            and re.fullmatch(r"[가-힣]{2,}[시군구]", department[1]) is None
        )
        if meta.get("region_scope") == "sido" or province_department:
            province_policies.setdefault(sido, set()).add(row["policy_id"])


def requested_province_ids(question, sido):
    if not sido:
        return None
    alias = re.sub(r"(특별자치|특별|광역)(시|도)$", r"\2", sido)
    names = "|".join(re.escape(name) for name in dict.fromkeys((sido, alias)))
    if re.search(rf"(?:{names})\s*(?:전체|전역|차원|단위)", question):
        return sorted(province_policies.get(sido, ()))
    return None

## 2. A안 · Qdrant 단독

거주지 기준 질문에는 기존 지역 필터를 적용한다. 질문에서 광역시·도 전체를 명시하면 해당 광역기관 소관 정책을 검색한다.
광역기관 소관은 검색 후보를 좁히는 기준이며, 세부 신청 자격과 지역 범위는 정책 본문에서 확인한다.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from qdrant_search import COLLECTION_NAME, embed_query, get_embedding, search_policies

configured_collection = os.getenv("QDRANT_COLLECTION") or COLLECTION_NAME
if configured_collection != COLLECTION_NAME:
    raise RuntimeError(f"A_fixed v1 평가의 QDRANT_COLLECTION은 {COLLECTION_NAME}이어야 합니다.")

qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL") or "http://localhost:6333",
    api_key=os.getenv("QDRANT_API_KEY") or None,
    timeout=30,
)
if not qdrant_client.collection_exists(configured_collection):
    raise RuntimeError(f"Qdrant에 {configured_collection}이 없습니다. 1절의 적재를 먼저 진행하세요.")
collection = qdrant_client.get_collection(configured_collection)
point_count = qdrant_client.count(configured_collection, exact=True).count
vectors = collection.config.params.vectors
if (
    not isinstance(vectors, VectorParams)
    or vectors.size != embedding_info["model"]["dim"]
    or vectors.distance != Distance.COSINE
    or point_count != embedding_info["n_chunks"]
):
    raise RuntimeError(f"{configured_collection}의 벡터 설정 또는 포인트 수가 A_fixed v1과 다릅니다.")

print(f"Qdrant {configured_collection}: {point_count:,}개", flush=True)
vectorstore = QdrantVectorStore(
    client=qdrant_client, collection_name=configured_collection, embedding=get_embedding(),
    validate_collection_config=False,
)


def retrieve_a(question, sido, sigungu, k):
    hits = search_policies(
        question, sido, sigungu, k * 3, vectorstore=vectorstore,
        parent_ids=requested_province_ids(question, sido),
    )
    return E.dedup_top(hits, k)

## 3. B안 · Neo4j 단독

`RagABFixedChunk.embedding`의 `rag_a_fixed_kure_v1_embedding` 인덱스를 Cypher 25의 `SEARCH`로 조회한다.
A와 같은 지역 조건을 적용한다. 광역 정책을 명시한 질문은 후보 수를 충분히 확보한 뒤 동일한 정책 ID로 거른다.

In [ ]:
NEO4J_INDEX = "rag_a_fixed_kure_v1_embedding"
CYPHER_B = f'''
CYPHER 25
MATCH (node:RagABFixedChunk)
SEARCH node IN (VECTOR INDEX {NEO4J_INDEX} FOR $qvec LIMIT $n) SCORE AS score
WHERE ($parent_ids IS NOT NULL AND node.parent_id IN $parent_ids)
   OR ($parent_ids IS NULL AND ($sido IS NULL OR (
       (node.sido_name IS NULL OR node.sido_name = $sido)
       AND ($sigungu IS NULL OR node.sigungu_name IS NULL OR node.sigungu_name = $sigungu)
   )))
RETURN node.parent_id AS parent_id, node.chunk_id AS chunk_id, score, node.text AS text,
       node.record_type AS record_type
ORDER BY score DESC, chunk_id
LIMIT $limit
'''


def retrieve_b(question, sido, sigungu, k, *, driver, database):
    parent_ids = requested_province_ids(question, sido)
    if parent_ids == []:
        return []
    total = embedding_info["n_chunks"]
    records = driver.execute_query(
        CYPHER_B,
        n=total if parent_ids is not None else min(k * 20, total),
        qvec=embed_query(question),
        parent_ids=parent_ids,
        sido=sido or None,
        sigungu=sigungu or None,
        limit=k * 3,
        database_=database,
        routing_="r",
    ).records
    return E.dedup_top([{**row.data(), "source": "neo4j"} for row in records], k)

## 4. C안 · Qdrant + Neo4j

두 벡터 검색에서 각각 상위 3k개의 부모 후보를 가져온다. `parent_id`로 합친 뒤, 각 검색의 순위 r에 대해 `1 / (60 + r)`을 더하는 RRF로 최종 k개를 고른다.
한쪽에만 나온 후보도 포함된다. 점수가 같으면 부모 ID 순으로 정렬한다.
선택된 청크와 부모가 Neo4j에서 일치하면 출처를 붙인다.
문항별 CSV의 `ranked_parent_ids`와 `fusion_ranks`에는 최종 결과 순서와 각 결과의 [Qdrant 순위, Neo4j 순위]가 기록된다.

In [ ]:
def retrieve_c(question, sido, sigungu, k, *, graph, neo4j_search):
    candidate_limit = k * 3
    qdrant_hits = retrieve_a(question, sido, sigungu, candidate_limit)
    neo4j_hits = neo4j_search(question, sido, sigungu, candidate_limit)
    candidates = {}
    for rank_key, hits in (("qdrant_rank", qdrant_hits), ("neo4j_rank", neo4j_hits)):
        for rank, hit in enumerate(hits, start=1):
            row = candidates.setdefault(
                hit["parent_id"], {**hit, "qdrant_rank": None, "neo4j_rank": None}
            )
            row[rank_key] = rank

    for row in candidates.values():
        ranks = [rank for rank in (row["qdrant_rank"], row["neo4j_rank"]) if rank is not None]
        row["score"] = sum(1 / (60 + rank) for rank in ranks)
        if len(ranks) == 2:
            row["source"] = "qdrant+neo4j"

    ranked = sorted(candidates.values(), key=lambda row: (-row["score"], row["parent_id"]))[:k]
    evidence = {
        row["chunk_id"]: row for row in graph.get_chunk_evidence([row["chunk_id"] for row in ranked])
    }
    for row in ranked:
        match = evidence.get(row["chunk_id"])
        if match and match["parent_id"] == row["parent_id"]:
            row["neo4j_evidence"] = match
    return ranked

## 5. 채점 실행

선택한 검색 방식에 동일한 질문을 넣고 검색 시간도 기록한다. CPU에서는 수 분이 걸릴 수 있다.

In [ ]:
from contextlib import ExitStack
from functools import partial

with ExitStack() as stack:
    systems = {"A_qdrant": retrieve_a}
    if USE_B or USE_C:
        from src.retrieval.neo4j_retriever import Neo4jConfig, Neo4jRetriever

        config = Neo4jConfig.from_env()
        graph = stack.enter_context(Neo4jRetriever(config))
        driver = graph.driver
        database = config.database
        counts = driver.execute_query(
            "MATCH (c:RagABFixedChunk) RETURN c.record_type AS kind, count(c) AS n",
            database_=database,
            routing_="r",
        ).records
        if {row["kind"]: row["n"] for row in counts} != embedding_info["counts_by_type"]:
            raise RuntimeError("Neo4j에 A_fixed v1 청크를 먼저 적재하거나 덤프를 복원하세요.")

        indexes = driver.execute_query(
            "SHOW VECTOR INDEXES YIELD name, state WHERE name = $name RETURN state",
            name=NEO4J_INDEX,
            database_=database,
            routing_="r",
        ).records
        if not indexes or indexes[0]["state"] != "ONLINE":
            raise RuntimeError(f"Neo4j에서 {NEO4J_INDEX} 인덱스가 ONLINE인지 확인하세요.")

        neo4j_search = partial(retrieve_b, driver=driver, database=database)
        if USE_B:
            systems["B_neo4j"] = neo4j_search
        if USE_C:
            systems["C_hybrid"] = partial(retrieve_c, graph=graph, neo4j_search=neo4j_search)

    evaluated = []
    for name, retrieve in systems.items():
        print(f"{name} · {len(questions)}문항 평가 중", flush=True)
        evaluated.append(E.run_eval(retrieve, questions, name))

detail = pd.concat(evaluated, ignore_index=True)
summary = E.summarize(detail)
summary

## 6. 결과 보기
- **v3**: 단일 질문 Hit@5 · MRR@5 → 뜻으로 찾는 기본 정확도
- **abc**: 묶음 Recall@10 → 모으기 · 넓히기 · 잇기 (그래프가 잘할 것으로 예상)
- **ragas**: 페르소나 · 질문 종류별 약점

In [ ]:
abc = detail[detail["set"] == "abc"]
print("abc 유형별 Recall@10")
display(abc.pivot_table(index="q_type", columns="system", values="recall10", aggfunc="mean").round(3))

rg = detail[detail["set"] == "ragas"].copy()
rg["score"] = rg["rr"].fillna(rg["recall10"])
print("ragas 페르소나 × 질문 종류 (단순=MRR@5, 복잡·추상=Recall@10)")
display(rg.pivot_table(index=["persona", "q_type"], columns="system", values="score", aggfunc="mean").round(3))

print("검색 결과로 채점할 수 있는 함정 문항")
display(detail[detail["mode"] == "trap"].pivot_table(index=["set", "qid"], columns="system", values="trap_pass"))

print("함정 미채점 및 평가 제외 문항")
display(
    detail.loc[detail["scoring_note"].notna() & detail["trap_pass"].isna(),
               ["set", "qid", "scoring_note"]].drop_duplicates()
)

## 7. 저장
- 결과 파일을 팀에 공유한다. 실행할 때마다 시각이 붙은 새 파일로 저장된다(덮어쓰기 없음).

In [ ]:
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
results_dir = kit_root / "results"
results_dir.mkdir(exist_ok=True)
detail_path = results_dir / f"abc_detail_{stamp}.csv"
summary_path = results_dir / f"abc_summary_{stamp}.csv"

detail.to_csv(detail_path, index=False, encoding="utf-8-sig")
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("저장:", summary_path, detail_path, sep="\n")